# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [14]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start,
        'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    for attempt in range(1, tries + 1):
        try:
            r = requests.get(BASE, params=params, timeout=10)
        except requests.exceptions.RequestException as exc:
            if attempt == tries:
                raise RuntimeError(f"Network request failed after {tries} attempts: {exc}") from exc
            time.sleep(attempt)
            continue

        if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])

        if r.status_code not in RETRY_STATUSES:
            r.raise_for_status()

        if attempt < tries:
            time.sleep(attempt)
        else:
            r.raise_for_status()

    raise RuntimeError(f"Failed to fetch weather after {tries} attempts.")

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [15]:
used_fallback = False

try:
    raw_df = get_weather('2026-09-05', '2026-10-03')
    wx = raw_df[raw_df['time'].isin(['2026-09-05', '2026-10-03'])].copy()
except Exception as err:
    print(f"API call failed ({err}); switching to fallback.")
    used_fallback = True
    wx = None

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [16]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback or wx is None:
    used_fallback = True
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
else:
    wx['source'] = 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [17]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})

wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']].reset_index(drop=True)
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [18]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [19]:
d1 = wx.loc[wx['date'] == '2026-09-05'].iloc[0]
d2 = wx.loc[wx['date'] == '2026-10-03'].iloc[0]

diff = abs(d1['precip_mm'] - d2['precip_mm'])

if d1['precip_mm'] > d2['precip_mm']:
    wetter, drier = d1, d2
else:
    wetter, drier = d2, d1

print(
    f"{wetter['date'].strftime('%Y-%m-%d')} was wetter than "
    f"{drier['date'].strftime('%Y-%m-%d')} by {diff:.1f} mm of rainfall."
)

2026-10-03 was wetter than 2026-09-05 by 12.2 mm of rainfall.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [20]:
from datetime import datetime

print(f"Endpoint:    {BASE}")
print(f"Coordinates: 38.03 N, -78.51 W (Charlottesville, VA)")
print(f"Date Range:  2026-09-05 to 2026-10-03")
print(f"Retrieved:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Data Status: {'Fallback sample' if used_fallback else 'Live API'} ({wx['source'].iloc[0]})")

Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03 N, -78.51 W (Charlottesville, VA)
Date Range:  2026-09-05 to 2026-10-03
Retrieved:   2026-10-11 20:09:45
Data Status: Live API (open-meteo)


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame instead of raising an error, a left join against all 200 days of sales data would produce NaN values for all weather columns. Sometimes scripts will autofill these NaN with 0, so every single day would appear to have had 0.0 mm of rainfall.

A scatter plot of rain vs. revenue would display a single line cluster of 200 points at x = 0. Nobody would catch the bug because the pipeline ran without raising an exception, and the chart would show a plausible real-world finding: it just didn't rain during that period.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [21]:
wx_cville = wx.copy()
wx_cville['city'] = 'Charlottesville'

try:
    raw_orlando = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
    wx_orlando = raw_orlando[raw_orlando['time'].isin(['2026-09-05', '2026-10-03'])].copy()
    wx_orlando = wx_orlando.rename(columns={
        'time': 'date',
        'temperature_2m_max': 'temp_max',
        'precipitation_sum': 'precip_mm'
    })
    wx_orlando['date'] = pd.to_datetime(wx_orlando['date'])
    wx_orlando['source'] = 'open-meteo'
except Exception:
    wx_orlando = pd.DataFrame({
        'date': pd.to_datetime(['2026-09-05', '2026-10-03']),
        'temp_max': [32.0, 28.5],
        'precip_mm': [4.8, 15.2],
        'source': 'fallback sample'
    })

wx_orlando['city'] = 'Orlando'

combined_wx = pd.concat([wx_cville, wx_orlando], ignore_index=True)
combined_wx = combined_wx[['city', 'date', 'temp_max', 'precip_mm', 'source']]
combined_wx

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
